In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
sys.path.append('../../pattern_recognition/src')

from data import GraphMatrixDataset, CNNMatrixDataset
from utils import P300Getter, train_model, plot_sample, show_progress, validate_model, infer_model
from interpretation import *
from models_cnn import *
from models_gnn import *
from graph import get_delaunay_graph, get_pos_init_graph, plot_graph, get_neighbors_graph

In [4]:
import mne
import pandas as pd
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from torch import nn, optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F

from scipy.spatial import Delaunay
import networkx as nx
import scipy.sparse as sp
import time
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from statsmodels.stats.proportion import proportion_confint

from mne import Epochs, pick_types, events_from_annotations
from mne.channels import make_standard_montage, DigMontage
from mne.io import concatenate_raws, read_raw_edf
from mne.datasets import eegbci
from mne.decoding import Scaler

from torch_geometric.data import Data, InMemoryDataset

In [5]:
def standardize_per_sample(X, eps=1e-8):
    """
    Standardize each sample (row) of X to zero mean and unit variance.
    X: array-like of shape (n_samples, n_timestamps)
    """
    X = np.asarray(X, dtype=np.float32)
    mean = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, keepdims=True)  # population std (ddof=0)
    return (X - mean) / np.maximum(std, eps)

In [6]:
@torch.no_grad()
def borderline_smote_torch(
    X: torch.Tensor,
    y: torch.Tensor,
    minority_label: int = 1,
    k: int = 5,          # minority neighbors for synthesis
    m: int = 10,         # neighbors for "danger" detection
    ratio: float = 1.0,  # desired minority after oversampling
    generator: torch.Generator | None = None,
    standardize_for_neighbors: bool = True,
    eps: float = 1e-6
): 
    """
    Parameters
    - X: (n_samples, n_features) float tensor
    - y: (n_samples,) 1D tensor with binary labels (minority_label vs others)
    - minority_label: label value considered minority (default 1)
    - k: number of nearest minority neighbors used to synthesize (default 5)
    - m: number of nearest overall neighbors to detect "danger" (default 10)
    - ratio: target minority:majority ratio after oversampling (1.0 => equalize)
    - generator: torch.Generator for reproducibility (optional)
    - standardize_for_neighbors: z-score features for distance computations only
    - eps: small constant to avoid division by zero in std

    Returns
    - X_aug: (n_samples + n_synth, n_features) tensor (same dtype/device as X)
    - y_aug: (n_samples + n_synth,) tensor (same dtype/device as y)
    """
    device = X.device
    dtype = X.dtype

    # Ensure y is a 1D tensor
    y = y.view(-1)
    n, d = X.shape

    # Indices for classes
    y_bool_min = (y == minority_label)
    idx_min = torch.nonzero(y_bool_min, as_tuple=False).view(-1)
    idx_maj = torch.nonzero(~y_bool_min, as_tuple=False).view(-1)

    n_min = idx_min.numel()
    n_maj = idx_maj.numel()

    if n_min == 0 or n_maj == 0:
        return X, y  # nothing to do

    # Determine how many synthetic samples to generate
    target_n_min = math.ceil(ratio * n_maj)
    num_new = max(0, target_n_min - n_min)
    if num_new == 0:
        return X, y

    # Optionally standardize for neighbor search (but generate in original space)
    if standardize_for_neighbors:
        mean = X.mean(dim=0, keepdim=True)
        std = X.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
        X_std = (X - mean) / std
    else:
        X_std = X

    X_min = X[idx_min]
    X_min_std = X_std[idx_min]

    # Danger detection: find m nearest overall neighbors to each minority sample
    # Compute distances from minority to all samples in standardized space
    m_neighbors = min(m, max(1, n - 1))
    dist_all = torch.cdist(X_min_std, X_std, p=2)  # (n_min, n)
    # Get m+1 smallest to include self, then drop self
    topk_k = min(m_neighbors + 1, n)
    topk_vals, topk_idx = torch.topk(dist_all, k=topk_k, largest=False, dim=1)

    # Remove self index per row
    # Build filtered neighbor indices excluding self, keep up to m_neighbors
    neigh_idx_list = []
    for row in range(n_min):
        row_idx = topk_idx[row]
        # Exclude self (index equals original global index)
        self_idx = idx_min[row]
        mask = row_idx != self_idx
        filtered = row_idx[mask]
        neigh_idx_list.append(filtered[:m_neighbors])

    # Stack into tensor (pad if some rows shorter, but we limited to m_neighbors so lengths are equal)
    neigh_idx = torch.stack(neigh_idx_list, dim=0)  # (n_min, m_neighbors)

    # Count majority labels among neighbors
    neighbor_labels = y[neigh_idx]  # (n_min, m_neighbors)
    maj_counts = (neighbor_labels != minority_label).sum(dim=1)

    # Borderline-1 "DANGER" criterion: more than half neighbors are majority, but not all
    half = math.ceil(m_neighbors / 2)
    danger_mask = (maj_counts >= half) & (maj_counts < m_neighbors)
    idx_danger_min = idx_min[danger_mask]

    # Fallback: if no danger points, use all minority points
    if idx_danger_min.numel() == 0:
        idx_danger_min = idx_min

    # Prepare minority-only neighbor search for synthesis
    # For each danger point, find k nearest minority neighbors (excluding self)
    X_danger_std = X_std[idx_danger_min]
    dist_min = torch.cdist(X_danger_std, X_min_std, p=2)  # (n_danger, n_min)
    k_neighbors = min(k, max(1, n_min - 1))
    # Get k+1 to include potential self
    topk_vals_min, topk_idx_min = torch.topk(dist_min, k=min(k_neighbors + 1, n_min),
                                             largest=False, dim=1)
    # Remove self per row and keep k_neighbors
    neigh_min_idx_list = []
    for row in range(idx_danger_min.numel()):
        row_idx = topk_idx_min[row]
        # map minority-local indices to global indices
        global_min_indices = idx_min[row_idx]
        self_global = idx_danger_min[row]
        mask = global_min_indices != self_global
        filtered = global_min_indices[mask]
        neigh_min_idx_list.append(filtered[:k_neighbors])

    neigh_min_idx = torch.stack(neigh_min_idx_list, dim=0)  # (n_danger, k_neighbors)

    # Distribute num_new across danger points
    n_danger = idx_danger_min.numel()
    base = num_new // n_danger
    rem = num_new - base * n_danger
    counts = torch.full((n_danger,), base, dtype=torch.long, device=device)
    if rem > 0:
        if generator is None:
            sel = torch.randperm(n_danger, device=device)[:rem]
        else:
            sel = torch.randperm(n_danger, generator=generator, device=device)[:rem]
        counts[sel] += 1

    # Generate synthetic samples by interpolation in original space
    synth_list = []
    # Prepare random source
    def rand_uniform(shape):
        if generator is None:
            return torch.rand(shape, device=device, dtype=dtype)
        else:
            return torch.rand(shape, generator=generator, device=device, dtype=dtype)

    for row in range(n_danger):
        c = counts[row].item()
        if c <= 0:
            continue
        xi = X[idx_danger_min[row]]  # original space
        neighbors_global = neigh_min_idx[row]
        if neighbors_global.numel() == 0:
            continue
        # choose random neighbors for each synthetic sample
        if generator is None:
            choice = torch.randint(low=0, high=neighbors_global.numel(), size=(c,), device=device)
        else:
            choice = torch.randint(low=0, high=neighbors_global.numel(), size=(c,), device=device, generator=generator)
        xj = X[neighbors_global[choice]]  # (c, d)
        r = rand_uniform((c, 1))  # (c, 1)
        xi_rep = xi.unsqueeze(0).expand(c, -1)  # (c, d)
        x_new = xi_rep + r * (xj - xi_rep)      # (c, d)
        synth_list.append(x_new)

    if len(synth_list) == 0:
        return X, y

    X_syn = torch.cat(synth_list, dim=0).to(device=device, dtype=dtype)
    y_syn = torch.full((X_syn.size(0),), fill_value=minority_label, dtype=y.dtype, device=device)

    X_aug = torch.cat([X, X_syn], dim=0)
    y_aug = torch.cat([y, y_syn], dim=0)
    return X_aug, y_aug

In [7]:
import glob

DATA_PATH = '../processed_data/'

data = dict()
labels = dict()
for file in glob.glob(DATA_PATH + 'S*_P300_PZ.csv'):
    subj = file.split('/')[-1][:5]
    mat_data = np.loadtxt(file, delimiter=',')
    data[subj] = standardize_per_sample(mat_data[:, 1:])
    labels[subj] = mat_data[:, 0]
    data[subj] = np.vstack(data[subj])
    labels[subj] = np.hstack(labels[subj])

In [8]:
# Define P300 time window (e.g., 250–500 ms after stimulus onset)
# Adjust indices based on your sampling rate (e.g., 250 Hz -> 62 to 125 samples)
p300_start = 62  # 250 ms
p300_end = 125   # 500 ms

# Dictionary to store results
results = dict()

for subj in data.keys():
    # Separate target and nontarget intervals
    target_data = data[subj][labels[subj] == 1]
    nontarget_data = data[subj][labels[subj] == 0]
    
    # Average across target and nontarget intervals
    avg_target = np.mean(target_data, axis=0)
    avg_nontarget = np.mean(nontarget_data, axis=0)
    
    # Compute mean amplitude in P300 window
    mean_target = np.mean(avg_target[p300_start:p300_end])
    mean_nontarget = np.mean(avg_nontarget[p300_start:p300_end])
    
    # Store results
    results[subj] = {
        'mean_target': mean_target,
        'mean_nontarget': mean_nontarget,
        'difference': mean_target - mean_nontarget
    }

# Example: Print results for each subject
for subj, res in results.items():
    print(f"{subj}: Target={res['mean_target']:.3f}, Nontarget={res['mean_nontarget']:.3f}, Diff={res['difference']:.3f}")


S0601: Target=-0.027, Nontarget=0.001, Diff=-0.028
S0201: Target=0.246, Nontarget=-0.010, Diff=0.256
S1201: Target=0.147, Nontarget=-0.011, Diff=0.158
S1601: Target=0.292, Nontarget=-0.028, Diff=0.320
S1801: Target=0.081, Nontarget=-0.005, Diff=0.087
S1901: Target=0.467, Nontarget=-0.024, Diff=0.491
S2001: Target=0.046, Nontarget=-0.025, Diff=0.071
S1701: Target=0.116, Nontarget=-0.006, Diff=0.122
S1401: Target=0.134, Nontarget=-0.012, Diff=0.146
S0701: Target=0.077, Nontarget=-0.011, Diff=0.088


In [9]:
min_len = min(arr.shape[0] for arr in labels.values())

for subj in data.keys():
    permutation = torch.randperm(min_len)
    data[subj] = torch.tensor(data[subj][:min_len]).float()[permutation]
    labels[subj] = torch.tensor(labels[subj].squeeze()[:min_len])[permutation]

In [10]:
all_train_data = []
all_train_labels = []
all_val_data = []
all_val_labels = []

for subj in data.keys():
    train_data, val_data, train_labels, val_labels = train_test_split(
        data[subj], 
        labels[subj], 
        test_size=0.15,
        shuffle=False
    )
    #train_data, train_labels = positive_upsample(train_data, train_labels)
    train_data, train_labels = borderline_smote_torch(train_data, train_labels, k=15, m=10, standardize_for_neighbors=False)
    all_train_data.append(train_data)
    all_train_labels.append(train_labels)
    all_val_data.append(val_data)
    all_val_labels.append(val_labels)

all_train_data = np.vstack(all_train_data)
all_train_labels = np.hstack(all_train_labels)
all_val_data = np.vstack(all_val_data)
all_val_labels = np.hstack(all_val_labels)

In [11]:
import numpy as np
from dataclasses import dataclass

@dataclass
class PeakPickingAverager:
    p300_window: tuple  # (start_idx, end_idx)
    baseline_end: int   # last index of baseline (exclusive)

    def fit(self, X, y):
        """
        X: (n_samples, n_timestamps)
        y: (n_samples,), 1 = target, 0 = nontarget
        """
        X = np.asarray(X)
        y = np.asarray(y)
        assert X.ndim == 2
        assert X.shape[0] == y.shape[0]

        self.classes_ = np.array([0, 1])

        # Class-wise averages (ERPs)
        self.avg_target_ = X[y == 1].mean(axis=0)
        self.avg_nontarget_ = X[y == 0].mean(axis=0)
        return self

    def _p300_amplitude(self, epoch):
        """
        Peak-to-peak: max in P300 window minus min in baseline.
        """
        start, end = self.p300_window
        baseline = epoch[:self.baseline_end]
        p300_seg = epoch[start:end]
        return float(p300_seg.max() - baseline.min())

    def decision_function(self, X):
        """
        Returns P300 score for each epoch: target-like minus nontarget-like.
        """
        X = np.asarray(X)
        scores = np.zeros(X.shape[0])
        for i, epoch in enumerate(X):
            # score relative to each class template
            s_t = self._p300_amplitude(epoch - self.avg_nontarget_)
            s_nt = self._p300_amplitude(epoch - self.avg_target_)
            scores[i] = s_t - s_nt
        return scores

    def predict_proba(self, X):
        scores = self.decision_function(X)
        # simple logistic squashing
        p1 = 1 / (1 + np.exp(-scores))
        return np.vstack([1 - p1, p1]).T

    def predict(self, X):
        proba = self.predict_proba(X)
        return (proba[:, 1] >= 0.5).astype(int)


## Общая модель

In [15]:
def get_metrics(preds, labels):
    corrects = (preds == labels).sum()
    min_acc, max_acc = proportion_confint(corrects, len(preds), 0.05)
    accuracy = corrects / len(preds)
    f1 = f1_score(labels, preds)
    return {'Accuracy': accuracy, 'F1-score': f1, 'Min Accuracy': min_acc, 'Max Accuracy': max_acc}

In [16]:
from sklearn.metrics import f1_score

p300_window = (62, 125) # 250–500 ms for 250 Hz
baseline_end = 62    # up to 250 ms
clf = PeakPickingAverager(p300_window, baseline_end).fit(all_train_data, all_train_labels)
preds = clf.predict(all_val_data)

common_acc = get_metrics(preds, all_val_labels)
print(f"Common Accuracy: {common_acc['Accuracy']:.3f}")
print(f"Common F1-Score: {common_acc['F1-score']:.3f}")
print(f'Common accuracy CI: [{round(common_acc["Min Accuracy"], 3)}, {round(common_acc["Max Accuracy"], 3)}]')

Common Accuracy: 0.187
Common F1-Score: 0.136
Common accuracy CI: [0.179, 0.194]


In [18]:
from sklearn.svm import SVC
from sklearn.metrics import f1_score

clf = SVC().fit(all_train_data, all_train_labels)
preds = clf.predict(all_val_data)

common_acc = get_metrics(preds, all_val_labels)
print(f"Common Accuracy: {common_acc['Accuracy']:.3f}")
print(f"Common F1-Score: {common_acc['F1-score']:.3f}")
print(f'Common accuracy CI: [{round(common_acc["Min Accuracy"], 3)}, {round(common_acc["Max Accuracy"], 3)}]')

Common Accuracy: 0.933
Common F1-Score: 0.086
Common accuracy CI: [0.928, 0.938]


## Индивидуальные модели

In [14]:
dataloaders = dict()
for subj in data.keys():
    train_data, val_data, train_labels, val_labels = train_test_split(
        data[subj], 
        labels[subj], 
        test_size=0.15,
        shuffle=False
    )
    temp_dataloaders = dict()
    train_data, train_labels = borderline_smote_torch(train_data, train_labels, k=15, m=10, standardize_for_neighbors=False)
    temp_dataloaders['train'] = np.array(train_data), np.array(train_labels)
    temp_dataloaders['val'] = np.array(val_data), np.array(val_labels)
    
    dataloaders[subj] = temp_dataloaders

In [17]:
result_sep = dict()
single_models = dict()
for subj, dataloader in dataloaders.items():
    clf = SVC().fit(*dataloader['train'])
    preds = clf.predict(dataloader['val'][0])
    
    print('Common model metrics:')
    print(f'{subj} accuracy: ', common_acc['Accuracy'])
    print(f'{subj} f1-score: ', common_acc['F1-score'])
    print(f'{subj} accuracy CI: [{round(common_acc["Min Accuracy"], 3)}, {round(common_acc["Max Accuracy"], 3)}]')
    acc = get_metrics(preds, dataloader['val'][1])
    print()
    print('Specific model metrics:')
    print(f'{subj} accuracy: ', acc['Accuracy'])
    print(f'{subj} f1-score: ', acc['F1-score'])
    print(f'{subj} accuracy CI: [{round(acc["Min Accuracy"], 3)}, {round(acc["Max Accuracy"], 3)}]')
    result_sep[subj] = {'accuracy': acc['Accuracy'], 
                        'f1-score': acc['F1-score'],
                        'size': len(dataloader['val'][1]),
                        'lower_ci': acc['Min Accuracy'],
                        'upper_ci': acc['Max Accuracy']}

Common model metrics:
S0601 accuracy:  0.18678500986193294
S0601 f1-score:  0.13618269432222918
S0601 accuracy CI: [0.179, 0.194]

Specific model metrics:
S0601 accuracy:  0.9250493096646942
S0601 f1-score:  0.02564102564102564
S0601 accuracy CI: [0.909, 0.941]
Common model metrics:
S0201 accuracy:  0.18678500986193294
S0201 f1-score:  0.13618269432222918
S0201 accuracy CI: [0.179, 0.194]

Specific model metrics:
S0201 accuracy:  0.9122287968441815
S0201 f1-score:  0.1523809523809524
S0201 accuracy CI: [0.895, 0.93]
Common model metrics:
S1201 accuracy:  0.18678500986193294
S1201 f1-score:  0.13618269432222918
S1201 accuracy CI: [0.179, 0.194]

Specific model metrics:
S1201 accuracy:  0.9230769230769231
S1201 f1-score:  0.1702127659574468
S1201 accuracy CI: [0.907, 0.939]
Common model metrics:
S1601 accuracy:  0.18678500986193294
S1601 f1-score:  0.13618269432222918
S1601 accuracy CI: [0.179, 0.194]

Specific model metrics:
S1601 accuracy:  0.9230769230769231
S1601 f1-score:  0.1702127

In [64]:
result_sep = dict()
single_models = dict()
for subj, dataloader in dataloaders.items():
    p300_window = (62, 125) # 250–500 ms for 250 Hz
    baseline_end = 62    # up to 250 ms
    clf = PeakPickingAverager(p300_window, baseline_end).fit(*dataloader['train'])
    preds = clf.predict(dataloader['val'][0])
    
    print('Common model metrics:')
    print(f'{subj} accuracy: ', common_acc['Accuracy'])
    print(f'{subj} f1-score: ', common_acc['F1-score'])
    print(f'{subj} accuracy CI: [{round(common_acc["Min Accuracy"], 3)}, {round(common_acc["Max Accuracy"], 3)}]')
    acc = get_metrics(preds, dataloader['val'][1])
    print()
    print('Specific model metrics:')
    print(f'{subj} accuracy: ', acc['Accuracy'])
    print(f'{subj} f1-score: ', acc['F1-score'])
    print(f'{subj} accuracy CI: [{round(acc["Min Accuracy"], 3)}, {round(acc["Max Accuracy"], 3)}]')
    result_sep[subj] = {'accuracy': acc['Accuracy'], 
                        'f1-score': acc['F1-score'],
                        'size': len(dataloader['val'][1]),
                        'lower_ci': acc['Min Accuracy'],
                        'upper_ci': acc['Max Accuracy']}

Common model metrics:
S0601 accuracy:  0.1873767258382643
S0601 f1-score:  0.13171759747102213
S0601 accuracy CI: [0.18, 0.195]

Specific model metrics:
S0601 accuracy:  0.47928994082840237
S0601 f1-score:  0.14285714285714285
S0601 accuracy CI: [0.449, 0.51]
Common model metrics:
S0201 accuracy:  0.1873767258382643
S0201 f1-score:  0.13171759747102213
S0201 accuracy CI: [0.18, 0.195]

Specific model metrics:
S0201 accuracy:  0.20907297830374755
S0201 f1-score:  0.13015184381778744
S0201 accuracy CI: [0.184, 0.234]
Common model metrics:
S1201 accuracy:  0.1873767258382643
S1201 f1-score:  0.13171759747102213
S1201 accuracy CI: [0.18, 0.195]

Specific model metrics:
S1201 accuracy:  0.3175542406311637
S1201 f1-score:  0.14987714987714987
S1201 accuracy CI: [0.289, 0.346]
Common model metrics:
S1601 accuracy:  0.1873767258382643
S1601 f1-score:  0.13171759747102213
S1601 accuracy CI: [0.18, 0.195]

Specific model metrics:
S1601 accuracy:  0.22583826429980275
S1601 f1-score:  0.1531823085

In [65]:
sum([value['accuracy'] * value['size'] for value in result_sep.values()]) / sum([value['size'] for value in result_sep.values()])

0.3526627218934911